# Cooperative Inverse Reinforcement Learning for Portfolio Management

**A Toy Study in Online Preference Learning**

### Problem

We consider a single risky asset with Gaussian returns and cash (risk-free, zero interest). The human's preferences are parameterized by θ = (λ_risk, λ_turn), where λ_risk penalizes return variance (risk aversion) and λ_turn penalizes trading activity (transaction costs or behavioral friction). The human makes decisions according to a Boltzmann-rational model: they are more likely to choose actions with higher utility, but not perfectly deterministic.

### CIRL
- both the agent and the human cooperate to achieve the human's goals
- the agent maintains a Bayesian belief over the human's preference parameters θ
- occasionally, the agent queries the human by presenting pairwise comparisons between candidate trading actions.
- the human's choice provides information about θ
- the agent updates its belief via Bayes' rule. 
- between queries, the agent trades greedily according to the current posterior mean estimate θ̂.
- model-based control with online Bayesian inference over the reward function
- **objective**: minimize regret (the gap between the agent's total reward and the oracle's total reward who knows true θ* at the start), minimize the number of queries used by the agent to get to oracle-grade performance
- testing: we compare the CIRL-based agent to a baseline that guesses θ or learns it offline



In [2]:
# Import dependencies
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Any

# Set random seed for reproducibility
np.random.seed(42)

#### Environment 

- **State**: `[price, shares, cash]`
  - `price`: Current price of the risky asset
  - `shares`: Number of shares held
  - `cash`: Cash holdings

- **Dynamics**: The risky asset has Gaussian returns:
  - r_t ~ Normal(μ, σ)
  - price_{t+1} = price_t × (1 + r_t)

- **Actions**: Discrete set {-1, 0, +1}
  - -1: Sell one share
  - 0: Hold
  - +1: Buy one share

- **Transaction Costs**: A small proportional cost applies to trades:
  - cost = trans_cost × |action| × price

- **Rewards**: return the change in portfolio value (PnL) as the raw reward, which will later be transformed according to the human's preferences.

This environment can be terminates in T steps.

In [7]:
class PortfolioEnv:
    """
    State: [price, shares, cash]
    Actions: {-1, 0, +1} (sell, hold, buy)
    Dynamics: Gaussian returns with transaction costs
    """
    
    def __init__(self, mu: float = 0.0005, sigma: float = 0.01, 
                 T: int = 100, trans_cost: float = 0.001,
                 init_price: float = 100.0, init_cash: float = 1000.0):
        """
            mu: Mean return of the risky asset per time step
            sigma: Standard deviation of returns
            T: Episode horizon (number of time steps)
            trans_cost: Proportional transaction cost (e.g., 0.001 = 0.1%)
            init_price: Initial price of the risky asset
            init_cash: Initial cash holdings
        """
        self.mu = mu
        self.sigma = sigma
        self.T = T
        self.trans_cost = trans_cost
        self.init_price = init_price
        self.init_cash = init_cash
        
        # State variables
        self.price = None
        self.shares = None
        self.cash = None
        self.t = None
        
    def reset(self) -> np.ndarray:
        """
        Reset the environment -> initial state.
        
        Returns:
            Initial state [price, shares, cash]
        """
        self.price = self.init_price
        self.shares = 0.0
        self.cash = self.init_cash
        self.t = 0
        return self._get_state()
    
    def _get_state(self) -> np.ndarray:
        return np.array([self.price, self.shares, self.cash])
    
    def _portfolio_value(self) -> float:
        return self.shares * self.price + self.cash
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict[str, Any]]:
        """
        One time step of the environment.
        """
        value_before = self._portfolio_value()
        
        cost = self.trans_cost * abs(action) * self.price
        self.cash -= cost
        
        trade_value = action * self.price
        self.shares += action
        self.cash -= trade_value
        
        if self.shares < 0 or self.cash < 0:
            # Revert trade if invalid
            self.shares -= action
            self.cash += trade_value
            self.cash += cost
            action = 0  # no op
        
        return_t = np.random.normal(self.mu, self.sigma)
        self.price = self.price * (1 + return_t)
        
        # PnL (change in portfolio value)
        value_after = self._portfolio_value()
        raw_pnl = value_after - value_before
        
        # Update time step
        self.t += 1
        done = (self.t >= self.T)
        
        info = {
            'time': self.t,
            'portfolio_value': value_after,
            'transaction_cost': cost,
            'return': return_t,
            'actual_action': action
        }
        
        return self._get_state(), raw_pnl, done, info

    def copy(self) -> 'PortfolioEnv':
        """
        Creates a copy of the environment
        """
        env_copy = PortfolioEnv(
            mu=self.mu, sigma=self.sigma, T=self.T, 
            trans_cost=self.trans_cost, init_price=self.init_price, 
            init_cash=self.init_cash
        )
        env_copy.price = self.price
        env_copy.shares = self.shares
        env_copy.cash = self.cash
        env_copy.t = self.t
        return env_copy

In [8]:
print('example environment set')

env = PortfolioEnv(mu=0.0005, sigma=0.01, T=10)
state = env.reset()

total_pnl = 0
# 5 steps example

for i in range(5):
    action = np.random.choice([-1, 0, 1])
    next_state, raw_pnl, done, info = env.step(action)
    total_pnl += raw_pnl
    
    print(f"Step {i+1}:")
    print(f"  Action: {action:+d} ({'Sell' if action == -1 else 'Hold' if action == 0 else 'Buy'})")
    print(f"  State: [price={next_state[0]:.2f}, shares={next_state[1]:.1f}, cash={next_state[2]:.2f}]")
    print(f"  Raw PnL: {raw_pnl:+.2f}")
    print(f"  Portfolio value: {info['portfolio_value']:.2f}")
    print(f"  Transaction cost: {info['transaction_cost']:.4f}")

example environment set
Step 1:
  Action: +1 (Buy)
  State: [price=101.06, shares=1.0, cash=899.90]
  Raw PnL: +0.96
  Portfolio value: 1000.96
  Transaction cost: 0.1000
Step 2:
  Action: +0 (Hold)
  State: [price=101.19, shares=1.0, cash=899.90]
  Raw PnL: +0.13
  Portfolio value: 1001.09
  Transaction cost: 0.0000
Step 3:
  Action: +0 (Hold)
  State: [price=101.08, shares=1.0, cash=899.90]
  Raw PnL: -0.11
  Portfolio value: 1000.98
  Transaction cost: 0.0000
Step 4:
  Action: +0 (Hold)
  State: [price=102.61, shares=1.0, cash=899.90]
  Raw PnL: +1.53
  Portfolio value: 1002.51
  Transaction cost: 0.0000
Step 5:
  Action: -1 (Sell)
  State: [price=104.24, shares=0.0, cash=1002.41]
  Raw PnL: -0.10
  Portfolio value: 1002.41
  Transaction cost: 0.1026


#### Human Pref. Model

##### Reward Function

**R(PnL, action; θ) = PnL - λ_risk × PnL² - λ_turn × 𝟙[action ≠ 0]**

- **PnL**: Raw profit/loss from the action
- **λ_risk**: Risk penalty (penalizes variance in returns)
- **λ_turn**: Turnover penalty (discourages frequent trading)

The quadratic risk term (PnL²) encourages smoother, more consistent returns. The turnover term penalizes any non-zero action.

##### Ground-Truth Preferences

θ* = (λ_risk*, λ_turn*) - the human's true preferences. 

##### Human Choice Model

The human chooses actions (given some candidates) according to a **Boltzmann-rational** model which simulates a tendency towards better actions but occasionally they make suboptimal choices:

**P(action | candidates, state) ∝ exp(β × Q(state, action))**

- Q(state, action): expected utility of taking that action (approximated using the environment's expected return μ)
- β: an inverse temperature parameter controlling rationality. 
- Higher β -> the human is more deterministic; Lower β -> more random.


In [9]:
def trading_reward(raw_pnl: float, action: int, theta: np.ndarray) -> float:
    lambda_risk, lambda_turn = theta
    
    risk_penalty = lambda_risk * (raw_pnl ** 2)
    
    turnover_penalty = lambda_turn * (1 if action != 0 else 0)
    
    reward = raw_pnl - risk_penalty - turnover_penalty
    return reward

# human preferences
theta_star = np.array([0.1, 0.1])

print("Ground-truth preference:")
print(f"λ_risk*: {theta_star[0]}")
print(f"λ_turn*: {theta_star[1]} ")

Ground-truth preference:
λ_risk*: 0.1
λ_turn*: 0.1 


In [10]:
def compute_expected_q(state: np.ndarray, action: int, theta: np.ndarray, 
                       env: PortfolioEnv) -> float:
    """
    Expected Q-value for an action using deterministic lookahead.
    """
    # current state
    env_copy = env.copy()
    
    value_before = env_copy._portfolio_value()
    
    cost = env_copy.trans_cost * abs(action) * env_copy.price
    env_copy.cash -= cost
    
    trade_value = action * env_copy.price
    env_copy.shares += action
    env_copy.cash -= trade_value
    
    # Check for invalid trades
    if env_copy.shares < 0 or env_copy.cash < 0:

        return -1e6

    env_copy.price = env_copy.price * (1 + env_copy.mu)

    value_after = env_copy._portfolio_value()
    expected_pnl = value_after - value_before
    
    q_value = trading_reward(expected_pnl, action, theta)
    
    return q_value


def softmax_probabilities(q_values: np.ndarray, beta: float = 5.0) -> np.ndarray:
    """
    Compute softmax probabilities from Q-values (distribution over actions)
    """
    q_max = np.max(q_values)
    exp_q = np.exp(beta * (q_values - q_max))
    probs = exp_q / np.sum(exp_q)
    return probs


def human_choice(state: np.ndarray, candidates: List[int], 
                 theta_star: np.ndarray, env: PortfolioEnv, 
                 beta: float = 5.0) -> int:
    """
    Simulate a human's choice
    """
    q_values = np.array([
        compute_expected_q(state, action, theta_star, env)
        for action in candidates
    ])
    probs = softmax_probabilities(q_values, beta)
    chosen_idx = np.random.choice(len(candidates), p=probs)
    chosen_action = candidates[chosen_idx]
    
    return chosen_action

In [11]:
# Test human choice model
env_test = PortfolioEnv()
state_test = env_test.reset()

candidates_test = [-1, 0, 1]
print(f"\nState: [price={state_test[0]:.2f}, shares={state_test[1]:.1f}, cash={state_test[2]:.2f}]")
print(f"Candidates: {candidates_test}")
print(f"True preferences: λ_risk={theta_star[0]}, λ_turn={theta_star[1]}")

print("Expected Q-values under true preferences:")
for action in candidates_test:
    q_val = compute_expected_q(state_test, action, theta_star, env_test)
    print(f"Action {action:+d}: Q = {q_val:.4f}")

print("Simulate 1000 human choices (β=4.0):")
choices = [human_choice(state_test, candidates_test, theta_star, env_test, beta=4.0) 
           for _ in range(1000)]
for action in candidates_test:
    count = choices.count(action)
    print(f"Action {action:+d}: {count/10:.1f}% of choices")


State: [price=100.00, shares=0.0, cash=1000.00]
Candidates: [-1, 0, 1]
True preferences: λ_risk=0.1, λ_turn=0.1
Expected Q-values under true preferences:
Action -1: Q = -1000000.0000
Action +0: Q = 0.0000
Action +1: Q = -0.1503
Simulate 1000 human choices (β=4.0):
Action -1: 0.0% of choices
Action +0: 66.3% of choices
Action +1: 33.7% of choices


#### CIRL Agent: Belief, Queries, and Control

The CIRL agent does three ops:

1. **Belief Maintenance**: Maintain a probability distribution over a discrete grid of θ values.
2. **Bayesian Update**: When the human responds to a query, update the belief using Bayes' rule.
3. **Control**: Select trading actions by optimizing under the current posterior mean θ̂.

##### Belief Representation

Discretize the parameter space into a small grid that the agent maintains a probability distribution (belief) over. Example below with 9 hypotheses:
- λ_risk ∈ {0.0, 0.1, 0.2}
- λ_turn ∈ {0.0, 0.1, 0.2}

##### Bayesian Inference

If agent queries the human with candidates {a₁, a₂} and observes choice â, the likelihood of this observation under hypothesis θ_k is:

P(â | θ_k, state, candidates) = softmax_β(Q(state, â; θ_k))

The posterior is then:

P(θ_k | â) ∝ P(â | θ_k) × P(θ_k)

We normalize to obtain a valid probability distribution.

##### Control Policy

The agent computes θ̂ = E[θ | belief] (posterior mean) and selects actions greedily:

**a* = argmax_a Q(state, a; θ̂)**

In [12]:
def make_theta_grid(lambda_risk_values: List[float] = [0.0, 0.1, 0.2],
                    lambda_turn_values: List[float] = [0.0, 0.1, 0.2]) -> np.ndarray:
    """
    Create a grid of theta values (preference parameters). Returns array of shape (K, 2) where each row is [lambda_risk, lambda_turn]
    """
    grid = []
    for lambda_risk in lambda_risk_values:
        for lambda_turn in lambda_turn_values:
            grid.append([lambda_risk, lambda_turn])
    return np.array(grid)


def initialize_uniform_belief(theta_grid: np.ndarray) -> np.ndarray:
    """
    Initialize a uniform belief over the theta grid, K hypotheses.
    """
    K = len(theta_grid)
    return np.ones(K) / K

In [14]:
theta_grid = make_theta_grid()
print("Theta grid (preference parameter hypotheses):")
print("idx, λ_risk, λ_turn")

for i, theta in enumerate(theta_grid):
    print(f"{i:2d}, {theta[0]:6.2f}, {theta[1]:6.2f}")

initial_belief = initialize_uniform_belief(theta_grid)
print(f"\nInitial belief: uniform over {len(theta_grid)} hypotheses")
print(f"P(θ_k) = {initial_belief[0]:.4f} for all k")

Theta grid (preference parameter hypotheses):
idx, λ_risk, λ_turn
 0,   0.00,   0.00
 1,   0.00,   0.10
 2,   0.00,   0.20
 3,   0.10,   0.00
 4,   0.10,   0.10
 5,   0.10,   0.20
 6,   0.20,   0.00
 7,   0.20,   0.10
 8,   0.20,   0.20

Initial belief: uniform over 9 hypotheses
P(θ_k) = 0.1111 for all k


In [15]:
def softmax_Q(state: np.ndarray, candidates: List[int], theta: np.ndarray,
              env: PortfolioEnv, beta: float = 5.0) -> np.ndarray:
    """
    Compute softmax probabilities over candidate actions for a given theta. 
    The likelihood model: P(action | theta, state, candidates).
    """
    q_values = np.array([
        compute_expected_q(state, action, theta, env)
        for action in candidates
    ])
    probs = softmax_probabilities(q_values, beta)
    return probs


def bayes_update(belief: np.ndarray, thetas: np.ndarray, 
                 state: np.ndarray, candidates: List[int], 
                 chosen_action: int, env: PortfolioEnv, 
                 beta: float = 5.0) -> np.ndarray:
    """
    Perform Bayesian update of belief given human's choice.
    Uses Bayes' rule: P(θ | choice) ∝ P(choice | θ) × P(θ)
    Returns posterior belief over theta grid, shape (K,)
    """
    K = len(thetas)
    likelihoods = np.zeros(K)
    
    chosen_idx = candidates.index(chosen_action)
    
    # Compute likelihood P(chosen_action | theta_k) for each hypothesis
    for k in range(K):
        probs = softmax_Q(state, candidates, thetas[k], env, beta)
        likelihoods[k] = probs[chosen_idx]
    
    posterior = likelihoods * belief
    
    posterior_sum = np.sum(posterior)
    if posterior_sum > 0:
        posterior = posterior / posterior_sum
    else:
        posterior = belief
    
    return posterior


def expected_theta(belief: np.ndarray, thetas: np.ndarray) -> np.ndarray:
    theta_mean = np.sum(belief[:, np.newaxis] * thetas, axis=0)
    return theta_mean


def pick_trading_action(state: np.ndarray, env: PortfolioEnv,
                       belief: np.ndarray, thetas: np.ndarray,
                       action_space: List[int] = [-1, 0, 1]) -> int:
    theta_hat = expected_theta(belief, thetas)

    q_values = np.array([
        compute_expected_q(state, action, theta_hat, env)
        for action in action_space
    ])
    
    # agent chooses action with the highest Q value
    best_idx = np.argmax(q_values)
    best_action = action_space[best_idx]
    
    return best_action

In [16]:
# Test Bayesian update

env_test = PortfolioEnv()
state_test = env_test.reset()
belief_test = initialize_uniform_belief(theta_grid)

print(f"\nInitial belief (uniform):")
for i in range(min(3, len(theta_grid))):
    print(f"θ_{i}: λ_risk={theta_grid[i][0]:.1f}, λ_turn={theta_grid[i][1]:.1f}, P={belief_test[i]:.4f}")

# Simulate a query
candidates_test = [-1, 1]  # Buy vs Sell
human_choice_test = human_choice(state_test, candidates_test, theta_star, env_test)

print(f"\nQuery: Choose between actions {candidates_test}")
print(f"Human chose: {human_choice_test:+d}")

belief_updated = bayes_update(belief_test, theta_grid, state_test, 
                              candidates_test, human_choice_test, env_test)

print(f"\nUpdated belief:")
for i in range(min(3, len(theta_grid))):
    print(f"θ_{i}: λ_risk={theta_grid[i][0]:.1f}, λ_turn={theta_grid[i][1]:.1f}, P={belief_updated[i]:.4f}")

theta_hat = expected_theta(belief_updated, theta_grid)
print(f"\nPosterior mean: θ̂ = [{theta_hat[0]:.3f}, {theta_hat[1]:.3f}]")
print(f"True theta: θ* = [{theta_star[0]:.3f}, {theta_star[1]:.3f}]")


Initial belief (uniform):
θ_0: λ_risk=0.0, λ_turn=0.0, P=0.1111
θ_1: λ_risk=0.0, λ_turn=0.1, P=0.1111
θ_2: λ_risk=0.0, λ_turn=0.2, P=0.1111

Query: Choose between actions [-1, 1]
Human chose: +1

Updated belief:
θ_0: λ_risk=0.0, λ_turn=0.0, P=0.1111
θ_1: λ_risk=0.0, λ_turn=0.1, P=0.1111
θ_2: λ_risk=0.0, λ_turn=0.2, P=0.1111

Posterior mean: θ̂ = [0.100, 0.100]
True theta: θ* = [0.100, 0.100]


##### TODO's

1. Create a baseline agent that commits to a preference estimate without interating with a human. It should start with a uniform prior over θ (or a fixed guess), compute θ̂ = E[θ] under the initial belief, never updates this estimate (no queries), and trades greedily according to the fixed θ̂.
2. Set up the experiment: set up an oracle that knows true θ* (real human preference) so we will use this as a reference point, then run episode simulations: a. simulate a CIRL agent with periodic queries, b. simulate the baseline agent with no queries, c. simulate oracle behavior for reference
3. Each of the experiments above should return a final reward where we will use to compute our regret score for each agent.
4. regret = oracle's total reward - agent's total reward

##### TODO EXPERIMENTS

We should run experiments to compare the CIRL agent against the baseline and the oracle.

### Experimental Design

- Episodes: 100 independent episodes with different random seeds
- Horizon: T = 100 time steps per episode
- CIRL query frequency: Query every 5 steps
- Metrics:
  - Distribution of regrets (vs oracle)
  - Mean of regret

### Hypotheses

1. CIRL should achieve lower regret than baseline by actively learning θ through queries and approach oracle performance faster
2. Diminishing returns from queries? Later queries should provide less information gain as belief converges.